In [1]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = r"C:\Users\natra\Final year project"

# IMPORTANT:
# Use the corrected preprocessing data
DATA_DIR = os.path.join(
    BASE_DIR,
    "outputs",
    "veremi_attack_corrected"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs",
    "veremi_attack_model_tuned"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# RANDOM SEED
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# TRAINING PARAMETERS
# ============================================================

BATCH_SIZE = 256

EPOCHS = 50

LEARNING_RATE = 0.001


# ============================================================
# IMPORTANT
# ============================================================
#
# We are NOT using the previous aggressive class weight 4.37.
#
# The previous BiLSTM+Attention model predicted almost all
# samples as Attack.
#
# We first train without class_weight so the model learns
# the natural decision boundary.
#
# The validation set is then used to select the threshold.
#
# ============================================================

USE_CLASS_WEIGHTS = False


# ============================================================
# LOAD DATA
# ============================================================

print("=" * 80)
print("VeReMi NextGen - Tuned BiLSTM Attack Detector")
print("=" * 80)

print("\nData directory:")
print(DATA_DIR)

print("\nOutput directory:")
print(OUTPUT_DIR)


X_train = np.load(
    os.path.join(
        DATA_DIR,
        "X_attack_train.npy"
    )
)

y_train = np.load(
    os.path.join(
        DATA_DIR,
        "y_attack_train.npy"
    )
)

X_val = np.load(
    os.path.join(
        DATA_DIR,
        "X_attack_validation.npy"
    )
)

y_val = np.load(
    os.path.join(
        DATA_DIR,
        "y_attack_validation.npy"
    )
)

X_test = np.load(
    os.path.join(
        DATA_DIR,
        "X_attack_test.npy"
    )
)

y_test = np.load(
    os.path.join(
        DATA_DIR,
        "y_attack_test.npy"
    )
)


# ============================================================
# DATA INFORMATION
# ============================================================

print("\n")
print("=" * 80)
print("DATASET INFORMATION")
print("=" * 80)

print(
    "\nX_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "X_val:",
    X_val.shape
)

print(
    "y_val:",
    y_val.shape
)

print(
    "X_test:",
    X_test.shape
)

print(
    "y_test:",
    y_test.shape
)


# ============================================================
# CLASS DISTRIBUTION
# ============================================================

train_normal = int(
    np.sum(y_train == 0)
)

train_attack = int(
    np.sum(y_train == 1)
)

val_normal = int(
    np.sum(y_val == 0)
)

val_attack = int(
    np.sum(y_val == 1)
)

test_normal = int(
    np.sum(y_test == 0)
)

test_attack = int(
    np.sum(y_test == 1)
)


print("\nClass distribution:")

print(
    f"Train      -> Normal: {train_normal:,}, "
    f"Attack: {train_attack:,}"
)

print(
    f"Validation -> Normal: {val_normal:,}, "
    f"Attack: {val_attack:,}"
)

print(
    f"Test       -> Normal: {test_normal:,}, "
    f"Attack: {test_attack:,}"
)


# ============================================================
# BUILD BiLSTM
# ============================================================

print("\n")
print("=" * 80)
print("BUILDING TUNED BiLSTM MODEL")
print("=" * 80)


timesteps = X_train.shape[1]

n_features = X_train.shape[2]


model = Sequential(
    [

        # ----------------------------------------------------
        # First BiLSTM
        # ----------------------------------------------------

        Bidirectional(
            LSTM(
                128,
                return_sequences=True,
                name="lstm_128"
            ),
            name="bilstm_128"
        ),

        BatchNormalization(
            name="batch_norm_1"
        ),

        Dropout(
            0.25,
            name="dropout_1"
        ),


        # ----------------------------------------------------
        # Second BiLSTM
        # ----------------------------------------------------

        Bidirectional(
            LSTM(
                64,
                return_sequences=False,
                name="lstm_64"
            ),
            name="bilstm_64"
        ),

        BatchNormalization(
            name="batch_norm_2"
        ),

        Dropout(
            0.25,
            name="dropout_2"
        ),


        # ----------------------------------------------------
        # Classification head
        # ----------------------------------------------------

        Dense(
            64,
            activation="relu",
            name="dense_64"
        ),

        Dropout(
            0.25,
            name="dropout_3"
        ),

        Dense(
            1,
            activation="sigmoid",
            name="attack_probability"
        )
    ],
    name="Tuned_BiLSTM"
)


# ============================================================
# COMPILE
# ============================================================

optimizer = Adam(
    learning_rate=LEARNING_RATE
)

model.compile(
    optimizer=optimizer,

    loss="binary_crossentropy",

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        ),

        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)


# ============================================================
# SUMMARY
# ============================================================

model.summary()


# ============================================================
# CALLBACKS
# ============================================================

model_path = os.path.join(
    OUTPUT_DIR,
    "bilstm_tuned_best.keras"
)


callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),

    ModelCheckpoint(
        model_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


# ============================================================
# TRAIN
# ============================================================

print("\n")
print("=" * 80)
print("TRAINING")
print("=" * 80)


if USE_CLASS_WEIGHTS:

    total = (
        train_normal
        + train_attack
    )

    class_weight = {
        0: total / (
            2 * train_normal
        ),

        1: total / (
            2 * train_attack
        )
    }

    print(
        "\nUsing class weights:"
    )

    print(
        class_weight
    )

else:

    class_weight = None

    print(
        "\nClass weights: DISABLED"
    )

    print(
        "This prevents the previous "
        "Attack-prediction bias."
    )


history = model.fit(

    X_train,

    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    class_weight=class_weight,

    callbacks=callbacks,

    verbose=1
)


# ============================================================
# SAVE HISTORY
# ============================================================

history_df = pd.DataFrame(
    history.history
)

history_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "bilstm_tuned_training_history.csv"
    ),
    index=False
)


# ============================================================
# LOAD BEST MODEL
# ============================================================

print("\n")
print("=" * 80)
print("LOADING BEST MODEL")
print("=" * 80)


model = tf.keras.models.load_model(
    model_path
)


# ============================================================
# VALIDATION PROBABILITIES
# ============================================================

print("\n")
print("=" * 80)
print("VALIDATION THRESHOLD SEARCH")
print("=" * 80)


val_probability = model.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=1
).reshape(-1)


# ============================================================
# SEARCH THRESHOLD
# ============================================================

thresholds = np.arange(
    0.10,
    0.951,
    0.01
)


threshold_results = []


for threshold in thresholds:

    val_pred = (
        val_probability >= threshold
    ).astype(np.int32)


    acc = accuracy_score(
        y_val,
        val_pred
    )

    precision = precision_score(
        y_val,
        val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        val_pred,
        zero_division=0
    )


    threshold_results.append(
        {
            "threshold": float(
                threshold
            ),

            "accuracy": float(
                acc
            ),

            "precision": float(
                precision
            ),

            "recall": float(
                recall
            ),

            "f1": float(
                f1
            )
        }
    )


threshold_df = pd.DataFrame(
    threshold_results
)


# ============================================================
# FIND BEST VALIDATION THRESHOLDS
# ============================================================

best_accuracy_row = (
    threshold_df
    .sort_values(
        "accuracy",
        ascending=False
    )
    .iloc[0]
)

best_f1_row = (
    threshold_df
    .sort_values(
        "f1",
        ascending=False
    )
    .iloc[0]
)


print("\nBest validation accuracy:")

print(
    f"Threshold: "
    f"{best_accuracy_row['threshold']:.2f}"
)

print(
    f"Accuracy: "
    f"{best_accuracy_row['accuracy']:.4f}"
)

print(
    f"Precision: "
    f"{best_accuracy_row['precision']:.4f}"
)

print(
    f"Recall: "
    f"{best_accuracy_row['recall']:.4f}"
)

print(
    f"F1: "
    f"{best_accuracy_row['f1']:.4f}"
)


print("\nBest validation F1:")

print(
    f"Threshold: "
    f"{best_f1_row['threshold']:.2f}"
)

print(
    f"Accuracy: "
    f"{best_f1_row['accuracy']:.4f}"
)

print(
    f"Precision: "
    f"{best_f1_row['precision']:.4f}"
)

print(
    f"Recall: "
    f"{best_f1_row['recall']:.4f}"
)

print(
    f"F1: "
    f"{best_f1_row['f1']:.4f}"
)


# ============================================================
# IMPORTANT:
# USE F1-OPTIMAL THRESHOLD FOR FINAL MODEL
#
# This avoids choosing a threshold from the test set.
# ============================================================

FINAL_THRESHOLD = float(
    best_f1_row["threshold"]
)


print("\n")
print(
    "Final threshold selected from "
    "VALIDATION data:"
)

print(
    f"{FINAL_THRESHOLD:.2f}"
)


# ============================================================
# SAVE THRESHOLD RESULTS
# ============================================================

threshold_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "bilstm_tuned_threshold_analysis.csv"
    ),
    index=False
)


# ============================================================
# TEST PREDICTIONS
# ============================================================

print("\n")
print("=" * 80)
print("GENERATING TEST PREDICTIONS")
print("=" * 80)


y_probability = model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
).reshape(-1)


y_pred = (
    y_probability
    >= FINAL_THRESHOLD
).astype(
    np.int32
)


# ============================================================
# TEST METRICS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

pr_auc = average_precision_score(
    y_test,
    y_probability
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

tn, fp, fn, tp = cm.ravel()


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("FINAL TEST RESULTS")
print("=" * 80)


print(
    f"\nThreshold: {FINAL_THRESHOLD:.2f}"
)

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

print(
    f"ROC-AUC  : {roc_auc:.4f}"
)

print(
    f"PR-AUC   : {pr_auc:.4f}"
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

print("\n")
print("=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)


print(
    "\n                Predicted"
)

print(
    "              Normal  Attack"
)

print(
    f"Actual Normal "
    f"{tn:8d} "
    f"{fp:8d}"
)

print(
    f"Actual Attack "
    f"{fn:8d} "
    f"{tp:8d}"
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n")
print("=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)


report = classification_report(
    y_test,
    y_pred,
    target_names=[
        "Normal",
        "Attack"
    ],
    digits=4,
    zero_division=0
)

print(
    report
)


# ============================================================
# SAVE PREDICTIONS
# ============================================================

np.save(
    os.path.join(
        OUTPUT_DIR,
        "bilstm_tuned_test_probabilities.npy"
    ),
    y_probability
)

np.save(
    os.path.join(
        OUTPUT_DIR,
        "bilstm_tuned_test_predictions.npy"
    ),
    y_pred
)

np.save(
    os.path.join(
        OUTPUT_DIR,
        "bilstm_tuned_test_labels.npy"
    ),
    y_test
)


# ============================================================
# SAVE CONFUSION MATRIX
# ============================================================

cm_df = pd.DataFrame(
    cm,

    index=[
        "Actual_Normal",
        "Actual_Attack"
    ],

    columns=[
        "Predicted_Normal",
        "Predicted_Attack"
    ]
)

cm_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "bilstm_tuned_confusion_matrix.csv"
    )
)


# ============================================================
# SAVE CLASSIFICATION REPORT
# ============================================================

report_dict = classification_report(
    y_test,
    y_pred,
    target_names=[
        "Normal",
        "Attack"
    ],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(
    report_dict
).transpose()


report_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "bilstm_tuned_classification_report.csv"
    )
)


# ============================================================
# SAVE RESULTS JSON
# ============================================================

results = {

    "model":
        "Tuned BiLSTM",

    "dataset":
        "VeReMi NextGen Highway 2",

    "sequence_length":
        int(timesteps),

    "number_of_features":
        int(n_features),

    "batch_size":
        BATCH_SIZE,

    "epochs":
        EPOCHS,

    "learning_rate":
        LEARNING_RATE,

    "class_weights_used":
        USE_CLASS_WEIGHTS,

    "threshold":
        FINAL_THRESHOLD,

    "train_samples":
        int(len(y_train)),

    "validation_samples":
        int(len(y_val)),

    "test_samples":
        int(len(y_test)),

    "accuracy":
        float(accuracy),

    "precision":
        float(precision),

    "recall":
        float(recall),

    "f1_score":
        float(f1),

    "roc_auc":
        float(roc_auc),

    "pr_auc":
        float(pr_auc),

    "true_negative":
        int(tn),

    "false_positive":
        int(fp),

    "false_negative":
        int(fn),

    "true_positive":
        int(tp)
}


with open(
    os.path.join(
        OUTPUT_DIR,
        "bilstm_tuned_results.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )


# ============================================================
# FINAL
# ============================================================

print("\n")
print("=" * 80)
print("TRAINING AND EVALUATION COMPLETE")
print("=" * 80)


print("\nBest model:")
print(model_path)


print("\nFinal test metrics:")

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

print(
    f"ROC-AUC  : {roc_auc:.4f}"
)

print(
    f"PR-AUC   : {pr_auc:.4f}"
)


print("\nResults saved to:")
print(OUTPUT_DIR)


print("\n")
print("=" * 80)
print("DONE")
print("=" * 80)

VeReMi NextGen - Tuned BiLSTM Attack Detector

Data directory:
C:\Users\natra\Final year project\outputs\veremi_attack_corrected

Output directory:
C:\Users\natra\Final year project\outputs\veremi_attack_model_tuned


DATASET INFORMATION

X_train: (192504, 8, 11)
y_train: (192504,)
X_val: (98464, 8, 11)
y_val: (98464,)
X_test: (215760, 8, 11)
y_test: (215760,)

Class distribution:
Train      -> Normal: 170,479, Attack: 22,025
Validation -> Normal: 86,285, Attack: 12,179
Test       -> Normal: 191,540, Attack: 24,220


BUILDING TUNED BiLSTM MODEL


Model: "Tuned_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ bilstm_128 (Bidirectional)           │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_norm_1 (BatchNormalization)    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bilstm_64 (Bidirectional)            │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_norm_2 (BatchNormalization)    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_64 (Dense)                     │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ attack_probability (Dense)           │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)



TRAINING

Class weights: DISABLED
This prevents the previous Attack-prediction bias.
Epoch 1/50
752/752 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.9435 - auc: 0.9216 - loss: 0.1738 - precision: 0.8442 - recall: 0.6212
Epoch 1: val_loss improved from None to 0.09795, saving model to C:\Users\natra\Final year project\outputs\veremi_attack_model_tuned\bilstm_tuned_best.keras

Epoch 1: finished saving model to C:\Users\natra\Final year project\outputs\veremi_attack_model_tuned\bilstm_tuned_best.keras
752/752 ━━━━━━━━━━━━━━━━━━━━ 128s 150ms/step - accuracy: 0.9435 - auc: 0.9216 - loss: 0.1738 - precision: 0.8442 - recall: 0.6212 - val_accuracy: 0.9669 - val_auc: 0.9801 - val_loss: 0.0980 - val_precision: 0.9339 - val_recall: 0.7886 - learning_rate: 0.0010
Epoch 2/50
752/752 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.9770 - auc: 0.9868 - loss: 0.0695 - precision: 0.9303 - recall: 0.8641
Epoch 2: val_loss improved from 0.09795 to 0.09375, saving model to C:\Users\natra\Final yea

In [1]:
# ================================================================
# VeReMi NextGen - FINAL ATTACK DETECTION
# INFERENCE / DEMO
# ================================================================

import os
import json
import numpy as np
import tensorflow as tf
# ================================================================
# 1. CONFIGURATION
# ================================================================

BASE_DIR = r"C:\Users\natra\Final year project"

# ------------------------------------------------
# Trained model
# ------------------------------------------------

MODEL_PATH = os.path.join(
    BASE_DIR,
    "outputs",
    "veremi_attack_model_tuned",
    "bilstm_tuned_best.keras"
)

# ------------------------------------------------
# Final results file
# Contains the threshold selected from validation
# ------------------------------------------------

RESULTS_PATH = os.path.join(
    BASE_DIR,
    "outputs",
    "veremi_attack_model_tuned",
    "bilstm_tuned_results.json"
)

# ------------------------------------------------
# Test data
# ------------------------------------------------

DATA_DIR = os.path.join(
    BASE_DIR,
    "outputs",
    "veremi_attack_corrected"
)

X_TEST_PATH = os.path.join(
    DATA_DIR,
    "X_attack_test.npy"
)

Y_TEST_PATH = os.path.join(
    DATA_DIR,
    "y_attack_test.npy"
)


# ================================================================
# 2. DEMO SETTINGS
# ================================================================

# ------------------------------------------------
# Set a particular test sample here.
#
# Example:
# DEMO_INDEX = 100
#
# If None, the program automatically selects
# an attack sample for demonstration.
# ------------------------------------------------

DEMO_INDEX = None

# ------------------------------------------------
# Choose what type of sample to demonstrate:
#
# "attack"  -> automatically find an attack
# "normal"  -> automatically find a normal sample
# "specific" -> use DEMO_INDEX
# ------------------------------------------------

DEMO_TYPE = "attack"


# ================================================================
# 3. FEATURE NAMES
# ================================================================

FEATURE_NAMES = [
    "x",
    "y",
    "speed",
    "acceleration",
    "heading_sin",
    "heading_cos",
    "delta_speed",
    "delta_acceleration",
    "delta_heading",
    "displacement",
    "estimated_speed"
]


# ================================================================
# 4. LOAD FILES
# ================================================================

print("=" * 75)
print("VeReMi NextGen - Attack Detection Inference Demo")
print("=" * 75)


required_files = [
    MODEL_PATH,
    RESULTS_PATH,
    X_TEST_PATH,
    Y_TEST_PATH
]


for path in required_files:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"\nRequired file not found:\n{path}"
        )


print("\nAll required files found.")


# ================================================================
# 5. LOAD TRAINED MODEL
# ================================================================

print("\nLoading trained BiLSTM model...")

model = tf.keras.models.load_model(
    MODEL_PATH
)

print("Model loaded successfully.")

print(
    "Model name:",
    model.name
)


# ================================================================
# 6. LOAD FINAL THRESHOLD
# ================================================================

with open(
    RESULTS_PATH,
    "r",
    encoding="utf-8"
) as f:

    results = json.load(f)


FINAL_THRESHOLD = float(
    results["threshold"]
)


print(
    f"\nFinal decision threshold: "
    f"{FINAL_THRESHOLD:.2f}"
)


# ================================================================
# 7. LOAD TEST DATA
# ================================================================

print("\nLoading test data...")

X_test = np.load(
    X_TEST_PATH
)

y_test = np.load(
    Y_TEST_PATH
)


print("\nTest dataset:")
print(
    "X_test:",
    X_test.shape
)

print(
    "y_test:",
    y_test.shape
)


# ================================================================
# 8. SELECT DEMO SAMPLE
# ================================================================

if DEMO_TYPE == "specific":

    if DEMO_INDEX is None:

        raise ValueError(
            "Set DEMO_INDEX when DEMO_TYPE='specific'."
        )

    sample_index = int(
        DEMO_INDEX
    )


elif DEMO_TYPE == "attack":

    attack_indices = np.where(
        y_test == 1
    )[0]

    if len(attack_indices) == 0:

        raise ValueError(
            "No attack samples found."
        )

    # Select a sample from the attack class.
    #
    # Taking the middle one gives a deterministic
    # demonstration sample.

    sample_index = int(
        attack_indices[len(attack_indices) // 2]
    )


elif DEMO_TYPE == "normal":

    normal_indices = np.where(
        y_test == 0
    )[0]

    if len(normal_indices) == 0:

        raise ValueError(
            "No normal samples found."
        )

    sample_index = int(
        normal_indices[len(normal_indices) // 2]
    )


else:

    raise ValueError(
        "DEMO_TYPE must be "
        "'attack', 'normal', or 'specific'."
    )


# ================================================================
# 9. EXTRACT ONE SEQUENCE
# ================================================================

sample = X_test[
    sample_index
]

true_label = int(
    y_test[sample_index]
)


print("\n" + "=" * 75)
print("SELECTED SAMPLE")
print("=" * 75)

print(
    f"\nSample index : {sample_index}"
)

print(
    f"Sequence shape: {sample.shape}"
)

print(
    f"Actual label : "
    f"{'ATTACK' if true_label == 1 else 'NORMAL'}"
)


# ================================================================
# 10. VERIFY INPUT SHAPE
# ================================================================

if sample.ndim != 2:

    raise ValueError(
        f"Expected sample shape "
        f"(timesteps, features), "
        f"got {sample.shape}"
    )


timesteps = sample.shape[0]

features = sample.shape[1]


print(
    f"\nTimesteps : {timesteps}"
)

print(
    f"Features  : {features}"
)


# ================================================================
# 11. DISPLAY INPUT FEATURES
# ================================================================

print("\n" + "=" * 75)
print("INPUT SEQUENCE")
print("=" * 75)

print(
    "\nThe model receives 8 consecutive messages "
    "with 11 features per message.\n"
)

header = (
    f"{'Step':<8}"
    f"{'x':>10}"
    f"{'y':>10}"
    f"{'speed':>10}"
    f"{'acc':>10}"
    f"{'h_sin':>10}"
    f"{'h_cos':>10}"
    f"{'d_speed':>10}"
    f"{'d_acc':>10}"
    f"{'d_head':>10}"
    f"{'disp':>10}"
    f"{'est_speed':>12}"
)

print(header)

print("-" * len(header))


for t in range(timesteps):

    values = sample[t]

    print(
        f"{t+1:<8}"
        f"{values[0]:>10.3f}"
        f"{values[1]:>10.3f}"
        f"{values[2]:>10.3f}"
        f"{values[3]:>10.3f}"
        f"{values[4]:>10.3f}"
        f"{values[5]:>10.3f}"
        f"{values[6]:>10.3f}"
        f"{values[7]:>10.3f}"
        f"{values[8]:>10.3f}"
        f"{values[9]:>10.3f}"
        f"{values[10]:>12.3f}"
    )


# ================================================================
# 12. PREPARE MODEL INPUT
# ================================================================

# Model expects:
#
# (batch_size, 8, 11)
#
# Add batch dimension.

model_input = sample[
    np.newaxis,
    ...
].astype(
    np.float32
)


print("\nModel input shape:")
print(
    model_input.shape
)


# ================================================================
# 13. RUN INFERENCE
# ================================================================

print("\n" + "=" * 75)
print("RUNNING MODEL INFERENCE")
print("=" * 75)

probability = model.predict(
    model_input,
    verbose=0
)


# Convert shape (1,1) -> scalar

attack_probability = float(
    probability[0][0]
)


# ================================================================
# 14. APPLY DECISION THRESHOLD
# ================================================================

prediction = int(
    attack_probability >= FINAL_THRESHOLD
)


# ================================================================
# 15. DISPLAY RESULT
# ================================================================

print("\n" + "=" * 75)
print("ATTACK DETECTION RESULT")
print("=" * 75)

print(
    f"\nAttack Probability : "
    f"{attack_probability:.4f}"
)

print(
    f"Decision Threshold : "
    f"{FINAL_THRESHOLD:.4f}"
)

print(
    f"Actual Label       : "
    f"{'ATTACK' if true_label == 1 else 'NORMAL'}"
)

print(
    f"Predicted Label    : "
    f"{'ATTACK' if prediction == 1 else 'NORMAL'}"
)


# ================================================================
# 16. SIMPLE INTERPRETATION
# ================================================================

print("\n" + "=" * 75)
print("INTERPRETATION")
print("=" * 75)

if prediction == 1:

    print(
        "\n>>> ALERT: Potential V2X Attack Detected"
    )

    print(
        f">>> Attack probability = "
        f"{attack_probability * 100:.2f}%"
    )

else:

    print(
        "\n>>> STATUS: Normal V2X Behavior"
    )

    print(
        f">>> Attack probability = "
        f"{attack_probability * 100:.2f}%"
    )


# ================================================================
# 17. CHECK WHETHER PREDICTION IS CORRECT
# ================================================================

if prediction == true_label:

    print(
        "\n>>> Demo prediction matches ground truth."
    )

else:

    print(
        "\n>>> Demo prediction differs from ground truth."
    )


# ================================================================
# 20. FINAL DEMO SUMMARY
# ================================================================

print("\n" + "=" * 75)
print("FINAL DEMO SUMMARY")
print("=" * 75)

print(
    f"\nInput sequence       : "
    f"{timesteps} messages × {features} features"
)

print(
    "Model                : Tuned BiLSTM"
)

print(
    f"Attack probability   : "
    f"{attack_probability:.4f}"
)

print(
    f"Decision threshold   : "
    f"{FINAL_THRESHOLD:.4f}"
)

print(
    f"Final prediction     : "
    f"{'ATTACK' if prediction == 1 else 'NORMAL'}"
)

print(
    f"Ground truth         : "
    f"{'ATTACK' if true_label == 1 else 'NORMAL'}"
)

print("\nInference complete.")


VeReMi NextGen - Attack Detection Inference Demo

All required files found.

Loading trained BiLSTM model...
Model loaded successfully.
Model name: Tuned_BiLSTM

Final decision threshold: 0.38

Loading test data...

Test dataset:
X_test: (215760, 8, 11)
y_test: (215760,)

SELECTED SAMPLE

Sample index : 116422
Sequence shape: (8, 11)
Actual label : ATTACK

Timesteps : 8
Features  : 11

INPUT SEQUENCE

The model receives 8 consecutive messages with 11 features per message.

Step             x         y     speed       acc     h_sin     h_cos   d_speed     d_acc    d_head      disp   est_speed
------------------------------------------------------------------------------------------------------------------------
1            0.328   -12.483    -0.131    -0.210     1.159    -0.453    -0.896    -0.717    -0.190    -0.279      -0.077
2            0.428   -12.512     0.096    -0.343     1.160    -0.452     1.173    -0.169    -0.193    -0.316      -0.077
3            0.526   -12.540    -0.585